# Matrix Multiplication and Straussan Matrix Multiplication

In [ ]:
import math


# creating a square matrix of size n
# getting values at specified row and column indices
# dividing matrix into 4 n/2 x n/2 matrices
class matrix:
    def __init__(self):
        '''
        Initializes an empty matrix object.
        Inputs:
            None
        Returns:
            None
        '''
        self.matrix = None
        self.size = 0

    def create(self, list_of_lists):
        '''
        Creates a square matrix from a list of lists.
        Inputs:
            list_of_lists: 2d list containing the values of the matrix
        Returns:
            self: matrix object containing the values from list_of_lists
        '''
        if not len(list_of_lists) == len(list_of_lists[0]):
            raise ValueError(f"must be square matrix, it was a {list_of_lists} by {len(list_of_lists[0])} ")
        lengths = [len(list_of_lists[i]) for i in range(len(list_of_lists))]
        if not max(lengths) == min(lengths):
            raise ValueError("All rows must be the same length")
        self.matrix = list_of_lists
        self.size = len(list_of_lists)
        return self

    def get(self, row, column):
        '''
        Gets the value at a specified row and column of the matrix.
        Inputs:
            row: integer containing the row index
            column: integer containing the column index
        Returns:
            value at the specified row and column
        '''
        return self.matrix[row][column]

    def divide_into_quadrants(self):
        '''
        Divides a matrix into four smaller matrices.
        Inputs:
            None
        Returns:
            matrices: list containing the four quadrants in the order
                      A11, A12, A21, A22
        '''
        left_half = self.size//2
        right_half = self.size - left_half
        quadrants = [[],[],[],[]]

        # quadrants are in the order A11, A12, A21, A22
        for i in range(left_half):
            quadrants[0].append(self.matrix[i][0:left_half])
            quadrants[1].append(self.matrix[i][left_half:len(self.matrix[i])])

        for i in range(right_half):
            quadrants[2].append(self.matrix[i + left_half][0:left_half])
            quadrants[3].append(self.matrix[i + left_half][left_half:len(self.matrix[i])])

        matrices = []
        for quad in quadrants:
            matrices.append(matrix().create(quad))

        return matrices

    def add(self, other):
        '''
        Adds two matrices together.
        Inputs:
            other: matrix object to be added to the current matrix
        Returns:
            matrix object containing the sum of the two matrices
        '''
        return matrix().create([
            [self.matrix[i][j] + other.matrix[i][j]
             for j in range(len(self.matrix[0]))]
            for i in range(len(self.matrix))
        ])

    def subtract(self, other):
        '''
        Subtracts one matrix from another matrix.
        Inputs:
            other: matrix object to be subtracted from the current matrix
        Returns:
            matrix object containing the difference between the two matrices
        '''
        return matrix().create([
            [self.matrix[i][j] - other.matrix[i][j]
             for j in range(len(self.matrix[0]))]
            for i in range(len(self.matrix))
        ])

    def multiply(self, other):
        '''
        Multiplies two square matrices using standard matrix multiplication.
        Inputs:
            other: matrix object to be multiplied with the current matrix
        Returns:
            output: matrix object containing the product of the two matrices
        '''
        output = matrix()

        if self.size != other.size:
            raise ValueError("Matrices must be the same size")

        output.create([[0 for i in range(self.size)]
                       for j in range(self.size)])

        for i, row in enumerate(self.matrix):
            for j, val in enumerate(row):
                for k in range(self.size):
                    output.matrix[i][j] += self.matrix[i][k] * other.matrix[k][j]

        return output

    def padToPowerTwoSize(self):
        '''
        Pads a matrix with zeros so its size is the next power of two.
        Inputs:
            None
        Returns:
            None
        '''
        size = self.size
        padded_size = 2 ** math.ceil(math.log2(self.size))

        if padded_size == self.size:
            return

        for i in range(padded_size - self.size):
            self.matrix.append([0 for i in range(self.size)])

        self.size = padded_size

        for i in range(self.size):
            self.matrix[i].extend([0 for i in range(padded_size - size)])

        return

    def strassenMultiply(self, other):
        '''
        Multiplies two square matrices using Strassen's matrix multiplication algorithm.
        Inputs:
            other: matrix object to be multiplied with the current matrix
        Returns:
            matrix object containing the product of the two matrices
        '''
        self.padToPowerTwoSize()
        other.padToPowerTwoSize()

        if self.size == 1:
            return matrix().create([
                [self.matrix[0][0] * other.matrix[0][0]]
            ])

        A_split = self.divide_into_quadrants()
        B_split = other.divide_into_quadrants()

        M1 = A_split[0].add(A_split[3]).strassenMultiply(
            B_split[0].add(B_split[3]))
        M2 = A_split[2].add(A_split[3]).strassenMultiply(B_split[0])
        M3 = A_split[0].strassenMultiply(
            B_split[1].subtract(B_split[3]))
        M4 = A_split[3].strassenMultiply(
            B_split[2].subtract(B_split[0]))
        M5 = A_split[0].add(A_split[1]).strassenMultiply(B_split[3])
        M6 = A_split[2].subtract(A_split[0]).strassenMultiply(
            B_split[0].add(B_split[1]))
        M7 = A_split[1].subtract(A_split[3]).strassenMultiply(
            B_split[2].add(B_split[3]))

        C11 = M1.add(M4).subtract(M5).add(M7)
        C12 = M3.add(M5)
        C21 = M2.add(M4)
        C22 = M1.subtract(M2).add(M3).add(M6)

        C11C12 = [
            C11.matrix[i] + C12.matrix[i]
            for i in range(len(C11.matrix))
        ]
        C21C22 = [
            C21.matrix[i] + C22.matrix[i]
            for i in range(len(C22.matrix))
        ]

        return matrix().create(C11C12 + C21C22)

## Tests

In [13]:
# Tests for standard matrix multiplication
m1 = matrix().create([[1, 2],
                      [3, 4]])

m2 = matrix().create([[5, 6],
                      [7, 8]])

assert m1.multiply(m2).matrix == [[19, 22],
                                  [43, 50]]


# Test identity matrix
m1 = matrix().create([[1, 2],
                      [3, 4]])

identity = matrix().create([[1, 0],
                            [0, 1]])

assert m1.multiply(identity).matrix == [[1, 2],
                                        [3, 4]]


# Tests for Strassen multiplication
m1 = matrix().create([[1, 2],
                      [3, 4]])

m2 = matrix().create([[5, 6],
                      [7, 8]])

assert m1.strassenMultiply(m2).matrix == [[19, 22],
                                          [43, 50]]


# Test Strassen with a matrix that needs padding
m1 = matrix().create([[1, 2, 3],
                      [4, 5, 6],
                      [7, 8, 9]])

m2 = matrix().create([[9, 8, 7],
                      [6, 5, 4],
                      [3, 2, 1]])

result = m1.strassenMultiply(m2).matrix

assert result == [[30, 24, 18, 0],
                  [84, 69, 54, 0],
                  [138, 114, 90, 0],
                  [0, 0, 0, 0]]

print("All matrix multiplication tests passed!")

All matrix multiplication tests passed!


## Benchmarking


In [ ]:
import time



start = time.perf_counter()

end = time.perf_counter()


# Sequence Alignment

In [ ]:

def align_sequences(seq, comp_seq):
    '''
    Calculates the cost matrix for global alignment of DNA sequences using the Needleman-Wunsch algorithm.
    Inputs: 
        seq: string containing a sequence
        comp_seq: string containing a sequence to be aligned with seq
    Returns:
        cost_matrix: 2d list containing cost matrix for Needleman-Wunsch algorithm
    '''
    matrix = []
    # initializing matrix
    matrix.append([-2*i for i in range(len(seq)+1)])
    for i in [-2*(i+1) for i in range(len(comp_seq))]:
        matrix.append([i])
    print(matrix)
    # filling in matrix
    for i in range(1, len(comp_seq) + 1):
        for j in range(1, len(seq) + 1):
            if comp_seq[i-1] == seq[j-1]:
                score = 1
            else:
                score = -1
            diagonal = matrix[i-1][j-1] + score
            up = matrix[i-1][j] - 2
            left = matrix[i][j-1] - 2
            matrix[i].append(max(diagonal, up, left))
    return matrix

def print_optimal_alignment(cost_matrix, seq, comp_seq):
    '''
    Calculates the alignment using the cost matrix for global alignment of DNA sequences using the Needleman-Wunsch algorithm.
    Inputs: 
        cost_matrix: 2d list containing cost matrix for Needleman-Wunsch algorithm
        seq: string containing a sequence
        comp_seq: string containing a sequence to be aligned with seq
    Returns:
        aligned_seq: string showing seq after alignment
        aligned_comp_seq: string showing comp_seq after alignment
    '''
    i = len(comp_seq)
    j = len(seq)
    aligned_seq = []
    aligned_comp_seq = []
    while i > 0 or j > 0:
        if i > 0 and j > 0:
            if comp_seq[i - 1] == seq[j - 1]:
                score = 1
            else:
                score = -1
            if cost_matrix[i][j] == cost_matrix[i - 1][j - 1] + score:
                aligned_seq.append(seq[j - 1])
                aligned_comp_seq.append(comp_seq[i - 1])
                i -= 1
                j -= 1
                continue
        if i > 0 and cost_matrix[i][j] == cost_matrix[i - 1][j] - 2:
            aligned_seq.append("-")
            aligned_comp_seq.append(comp_seq[i - 1])
            i -= 1
            continue
        if j > 0 and cost_matrix[i][j] == cost_matrix[i][j - 1] - 2:
            aligned_seq.append(seq[j - 1])
            aligned_comp_seq.append("-")
            j -= 1
            continue
    # constructs the alignment backwards
    aligned_seq.reverse()
    aligned_comp_seq.reverse()
    aligned_seq = "".join(aligned_seq)
    aligned_comp_seq = "".join(aligned_comp_seq)
    print(aligned_seq)
    print(aligned_comp_seq)
    return aligned_seq, aligned_comp_seq


## Tests

In [15]:
# Test identical sequences
seq1 = "ACG"
seq2 = "ACG"

cost_matrix = align_sequences(seq1, seq2)

assert cost_matrix == [
    [0, -2, -4, -6],
    [-2, 1, -1, -3],
    [-4, -1, 2, 0],
    [-6, -3, 0, 3]
]

alignment = print_optimal_alignment(cost_matrix, seq1, seq2)

assert alignment == ("ACG", "ACG")


# Test sequences with a mismatch
seq1 = "ACG"
seq2 = "ATG"

cost_matrix = align_sequences(seq1, seq2)

assert cost_matrix == [
    [0, -2, -4, -6],
    [-2, 1, -1, -3],
    [-4, -1, 0, -2],
    [-6, -3, -2, 1]
]

alignment = print_optimal_alignment(cost_matrix, seq1, seq2)

assert alignment == ("ACG", "ATG")


# Test alignment requiring a gap
seq1 = "ACG"
seq2 = "AG"

cost_matrix = align_sequences(seq1, seq2)

assert cost_matrix == [
    [0, -2, -4, -6],
    [-2, 1, -1, -3],
    [-4, -1, 0, 0]
]

alignment = print_optimal_alignment(cost_matrix, seq1, seq2)

assert alignment == ("ACG", "A-G")


# Test single-character sequences
seq1 = "A"
seq2 = "A"

cost_matrix = align_sequences(seq1, seq2)

assert cost_matrix == [
    [0, -2],
    [-2, 1]
]

assert print_optimal_alignment(cost_matrix, seq1, seq2) == ("A", "A")


# Test single-character mismatch
seq1 = "A"
seq2 = "G"

cost_matrix = align_sequences(seq1, seq2)

assert cost_matrix == [
    [0, -2],
    [-2, -1]
]

assert print_optimal_alignment(cost_matrix, seq1, seq2) == ("A", "G")


print("All Needleman-Wunsch tests passed!")


seq1 = "GATTACA"
seq2 = "GCATGCU"

cost_matrix = align_sequences(seq1, seq2)

# Optimal Needleman-Wunsch score with:
# match = +1, mismatch = -1, gap = -2
assert cost_matrix[-1][-1] == -1

aligned1, aligned2 = print_optimal_alignment(
    cost_matrix, seq1, seq2
)

# An alignment must have equal-length output strings
assert len(aligned1) == len(aligned2)

# Removing gaps should reconstruct the original sequences
assert aligned1.replace("-", "") == seq1
assert aligned2.replace("-", "") == seq2

print("Alignment validity tests passed!")

[[0, -2, -4, -6], [-2], [-4], [-6]]
ACG
ACG
[[0, -2, -4, -6], [-2], [-4], [-6]]
ACG
ATG
[[0, -2, -4, -6], [-2], [-4]]
ACG
A-G
[[0, -2], [-2]]
A
A
[[0, -2], [-2]]
A
G
All Needleman-Wunsch tests passed!
[[0, -2, -4, -6, -8, -10, -12, -14], [-2], [-4], [-6], [-8], [-10], [-12], [-14]]
GATTACA
GCATGCU
Alignment validity tests passed!
